# Poker-Eval Library Demo

This notebook demonstrates the capabilities of the poker-eval library for:
- Hand evaluation
- Equity calculation
- Range analysis
- ICM calculations

## Setup

First, import the poker-eval Python bindings.

In [ ]:
import sys
sys.path.insert(0, '../bindings/python')
sys.path.insert(0, '../bindings/cython/python')

from pokereval import PokerEval
pe = PokerEval()
print("Poker-eval loaded successfully!")

## 1. Hand Evaluation

Evaluate the strength of poker hands.

In [ ]:
# Evaluate a 7-card hand (Texas Hold'em style)
hand = ['As', 'Ks']  # Hole cards: Ace-King suited
board = ['Qs', 'Js', 'Ts', '2h', '3d']  # Board: Royal flush!

result = pe.best('hi', hand, board)
print(f"Hand value: {result[0]}")
print(f"Hand type: {result[1][0]}")
print(f"Best 5 cards: {result[1][1:]}")

In [ ]:
# Compare different hands
hands_to_compare = [
    (['Ah', 'Ad'], ['Kh', 'Qh', 'Jh', '9c', '2d'], 'AA on K-Q-J board'),
    (['Kh', 'Qh'], ['Ah', 'Jh', 'Th', '9c', '2d'], 'KQs with flush draw'),
    (['7h', '7d'], ['7s', '7c', '2h', '3d', '4s'], 'Quad sevens!'),
]

for hand, board, desc in hands_to_compare:
    result = pe.best('hi', hand, board)
    print(f"{desc}: {result[1][0]} (value: {result[0]})")

## 2. Equity Calculation

Calculate win probabilities between hands.

In [ ]:
# Classic matchup: AA vs KK preflop
result = pe.poker_eval(
    game='holdem',
    pockets=[['Ah', 'Ad'], ['Kh', 'Kd']],
    board=[],
    dead=[]
)

print("AA vs KK Preflop:")
print(f"  Samples: {result['info'][0]}")
for i, player_result in enumerate(result['eval']):
    equity = player_result['ev'] / 10.0
    print(f"  Player {i+1}: {equity:.1f}% equity")

In [ ]:
# Equity on a specific board
result = pe.poker_eval(
    game='holdem',
    pockets=[['Ah', 'Kh'], ['Qc', 'Qd']],
    board=['Jh', 'Th', '2s'],
    dead=[]
)

print("AKs vs QQ on JhTh2s flop:")
print(f"  AKs (nut flush draw + gutshot): {result['eval'][0]['ev']/10:.1f}%")
print(f"  QQ (overpair): {result['eval'][1]['ev']/10:.1f}%")

## 3. Omaha Hands

Generate Omaha hand combinations from patterns.

In [ ]:
# Generate all AAxx hands (Omaha)
aa_hands = pe.omaha_hands('AAxx')
print(f"Number of AAxx combinations: {len(aa_hands)}")
print(f"First 5 examples: {aa_hands[:5]}")

In [ ]:
# Generate double-suited rundowns
ds_rundowns = pe.omaha_hands('JT98ds')
print(f"Number of JT98 double-suited: {len(ds_rundowns)}")
print(f"Examples: {ds_rundowns[:3]}")

## 4. Range Equity Calculation

Calculate equity between hand ranges.

In [ ]:
# Omaha range vs range
try:
    result = pe.calculate_range_equity(
        game_name='omaha',
        list_of_player_range_definitions=[
            ['AAxx'],  # Player 1: All AAxx hands
            ['KKxx']   # Player 2: All KKxx hands
        ],
        board_card_strings=[],
        use_montecarlo=True,
        iterations=5000
    )
    
    print("Omaha AAxx vs KKxx preflop:")
    print(f"  Matchups evaluated: {result['info'][0]}")
    print(f"  AAxx equity: {result['eval'][0]['ev']*100:.1f}%")
    print(f"  KKxx equity: {result['eval'][1]['ev']*100:.1f}%")
except Exception as e:
    print(f"Range equity calculation not available: {e}")

## 5. Multi-Way Equity

Calculate equity in multi-way pots with invested amounts.

In [ ]:
# 3-way all-in with different stack sizes
try:
    result = pe.calculate_multiway_equity(
        game_name='holdem',
        list_of_player_range_definitions=[
            ['AhAd'],  # Player 1: AA
            ['KsKc'],  # Player 2: KK
            ['QhQd']   # Player 3: QQ
        ],
        invested_amounts=[100, 100, 100],
        board_card_strings=[],
        use_montecarlo=True,
        iterations=10000
    )
    
    print("3-way: AA vs KK vs QQ")
    print(f"  Matchups: {result['matchups']}")
    for i, player in enumerate(result['players']):
        print(f"  Player {i+1}: {player['equity']*100:.1f}% equity, EV: ${player['ev']:.2f}")
except Exception as e:
    print(f"Multiway equity calculation not available: {e}")

## 6. Low Hand Evaluation (Hi/Lo Games)

Evaluate low hands for games like Omaha Hi/Lo.

In [ ]:
# Evaluate a low hand
hand = ['Ah', '2d']
board = ['3h', '4s', '5c', 'Kd', 'Qh']

# High hand
high_result = pe.best('hi', hand, board)
print(f"High: {high_result[1][0]}")

# Low hand (A-5 straight for low, wheel!)
low_result = pe.best('low', hand, board)
if low_result[1][0] != 'Nothing':
    print(f"Low: {low_result[1][0]} - {pe.card2string(low_result[1][1:])}")
else:
    print("No qualifying low hand")

## 7. Stud Hands

Generate 7-Card Stud hand patterns.

In [ ]:
# Generate stud hands with (AA) in the hole
try:
    stud_hands = pe.get_stud_hands('(AA)Kxxxx', game_total_cards=7)
    print(f"Stud hands with rolled-up Aces: {len(stud_hands)}")
    print(f"Example: {stud_hands[0] if stud_hands else 'None'}")
except Exception as e:
    print(f"Stud hand generation: {e}")

## 8. Card Conversion Utilities

Convert between card representations.

In [ ]:
# String to number
cards = ['As', 'Kh', 'Qd', 'Jc', 'Ts']
numbers = pe.string2card(cards)
print(f"Cards: {cards}")
print(f"Numbers: {numbers}")

# Number to string
back_to_strings = pe.card2string(numbers)
print(f"Back to strings: {back_to_strings}")

In [ ]:
# Full deck
deck = pe.deck()
print(f"Full deck ({len(deck)} cards):")
for suit_start in range(0, 52, 13):
    suit_cards = deck[suit_start:suit_start+13]
    print(f"  {suit_cards}")

## Summary

The poker-eval library provides comprehensive poker analysis:

- **Hand Evaluation**: Fast evaluation of 5-7 card hands
- **Equity Calculation**: Exact and Monte Carlo equity
- **Range Analysis**: Omaha and Stud hand generation
- **Multi-Way**: 3+ player equity calculations
- **Hi/Lo**: Support for split pot games

For more information, see the documentation at `docs/`.